In [2]:
from google.cloud import bigquery

client = bigquery.Client(project="funnel-analysis-project-503810")

query = """
    SELECT *
    FROM `funnel-analysis-project-503810.funnel_analysis.funnel_first_touch`
"""

df = client.query(query).to_dataframe()

print(df.shape)
df.head()

C:\Users\dell\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


(79193, 6)


,user_pseudo_id,t_page_view,t_view_item,t_add_to_cart,t_begin_checkout,t_purchase
0,1451078.8215732025,2020-11-07 03:03:38.823587+00:00,2020-11-07 03:04:24.373270+00:00,NaT,2020-11-09 11:01:31.401692+00:00,2020-11-09 11:08:26.841419+00:00
1,1714104.6929445672,2020-11-15 09:31:16.450023+00:00,NaT,NaT,NaT,NaT
2,2633877.1089384309,2020-11-15 09:35:48.460907+00:00,NaT,NaT,NaT,NaT
3,2836669.4584691910,2020-11-15 13:46:37.157410+00:00,2020-11-15 13:54:29.135417+00:00,NaT,2020-11-15 14:25:42.302852+00:00,NaT
4,3972359.8765083540,2020-11-15 21:01:42.230260+00:00,2020-11-15 21:01:55.407171+00:00,NaT,NaT,NaT


In [23]:
import pandas as pd

In [ ]:
print(df['t_page_view'].notna().sum())
print(df['t_view_item'].notna().sum())
print(df['t_add_to_cart'].notna().sum())
print(df['t_begin_checkout'].notna().sum())
print(df['t_purchase'].notna().sum())

79181
21440
2060
4219
1532


### Data Quality Note: `add_to_cart` → `begin_checkout` Anomaly

Step conversion for `add_to_cart` → `begin_checkout` came out >100%, which is 
impossible. Investigation:

- 4,219 users reached `begin_checkout`
- 3,164 of those (75%) had **no** `add_to_cart` event logged

**Likely explanations:** a "Buy Now" flow bypassing `add_to_cart`, or a 
client-side tracking gap where the `add_to_cart` event fails to fire while 
`begin_checkout` fires later successfully.

**Limitation:** Cannot be confirmed from event data alone — would need access 
to the live site (UI flow, session recordings/heatmaps) to distinguish a real 
"buy now" button from an instrumentation bug. Both produce identical symptoms 
in the log. Since this is a public sample dataset with no site access, the 
two explanations remain indistinguishable here.

**Handling:** Step-conversion for this transition is reported but flagged as 
unreliable. Overall conversion (vs. `page_view`) is treated as the primary 
metric where this ambiguity matters.

In [11]:
skip_stage = df['t_add_to_cart'].isna() & df['t_begin_checkout'].notna()
print(skip_stage.sum())

3164


In [24]:
stages = ['t_page_view','t_view_item','t_add_to_cart','t_begin_checkout','t_purchase']
counts = df[stages].notna().sum()


overall_conversion = (counts / counts['t_page_view']) * 100
step_conversion = (counts / counts.shift(1)) * 100

funnel = pd.DataFrame({
    'count': counts,
    'overall_conversion_%': overall_conversion.round(1),
    'step_conversion_%': step_conversion.round(1)
})

print(funnel)

                  count  overall_conversion_%  step_conversion_%
t_page_view       79181                 100.0                NaN
t_view_item       21440                  27.1               27.1
t_add_to_cart      2060                   2.6                9.6
t_begin_checkout   4219                   5.3              204.8
t_purchase         1532                   1.9               36.3
